In [2]:
import torch
import torchvision
import torchvision.transforms as transforms
import time

# -----------------------------
# Load Dataset (same as before)
# -----------------------------
transform = transforms.ToTensor()

train_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform
)

test_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform
)

In [4]:
# Faster subset
subset = torch.utils.data.Subset(train_dataset, range(5000))

train_loader = torch.utils.data.DataLoader(subset, batch_size=128, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64)

In [5]:
# -----------------------------
# Model Parameters (UPDATED)
# -----------------------------
input_size = 784
h1_size = 256   # increased
h2_size = 128   # increased
output_size = 10

W1 = torch.randn(input_size, h1_size, requires_grad=True)
b1 = torch.zeros(h1_size, requires_grad=True)

W2 = torch.randn(h1_size, h2_size, requires_grad=True)
b2 = torch.zeros(h2_size, requires_grad=True)

W3 = torch.randn(h2_size, output_size, requires_grad=True)
b3 = torch.zeros(output_size, requires_grad=True)

lr = 0.01
epochs = 3

In [7]:
# -----------------------------
# Training
# -----------------------------
start_time = time.time()

for epoch in range(epochs):
    total_loss = 0

    for images, labels in train_loader:

        x = images.reshape(-1, 784)

        # Forward
        z1 = x @ W1 + b1
        h1 = torch.relu(z1)

        z2 = h1 @ W2 + b2
        h2 = torch.relu(z2)

        logits = h2 @ W3 + b3

        # Loss
        loss = torch.nn.functional.cross_entropy(logits, labels)

        # Backward
        loss.backward()

        # Update
        with torch.no_grad():
            W1 -= lr * W1.grad
            b1 -= lr * b1.grad

            W2 -= lr * W2.grad
            b2 -= lr * b2.grad

            W3 -= lr * W3.grad
            b3 -= lr * b3.grad

        # Zero grad
        W1.grad.zero_()
        b1.grad.zero_()
        W2.grad.zero_()
        b2.grad.zero_()
        W3.grad.zero_()
        b3.grad.zero_()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

end_time = time.time()
training_time = end_time - start_time

Epoch 1, Loss: 403.9106
Epoch 2, Loss: 315.2730
Epoch 3, Loss: 243.4919


In [8]:
# -----------------------------
# Evaluation (Accuracy)
# -----------------------------
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:

        x = images.reshape(-1, 784)

        z1 = x @ W1 + b1
        h1 = torch.relu(z1)

        z2 = h1 @ W2 + b2
        h2 = torch.relu(z2)

        logits = h2 @ W3 + b3

        preds = torch.argmax(logits, dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = 100 * correct / total


# -----------------------------
# Final Results
# -----------------------------
print(f"\nFinal Accuracy: {accuracy:.2f}%")
print(f"Training Time: {training_time:.2f} seconds")


Final Accuracy: 57.33%
Training Time: 3.98 seconds
